In [ ]:
#@title 按這裡開始（先按 ▶）
# ←投影片未含，執行所需
print("✅ W16 出發！本週目標：自己寫混淆矩陣與三個指標，不用 sklearn 的現成函式")
print("sklearn 只拿來造資料與訓練，四個指標一律自己算")
print("本週要自己補七個空：my_confusion 兩行、三個指標三行、PR 曲線一行、乾淨訓練集一行")

# W16　模型評估與常見陷阱（電腦教室版）

**開始之前**：按「複製到雲端硬碟」，改自己的副本才存得起來。

筆記本是**填空式**：看到 `____` 就是你要動手的地方。卡住了先看該格上方的說明，
真的過不去再展開最後一格的「參考解」。

| 任務 | 你要自己寫的 | 產物 |
|---|---|---|
| 一 | （照跑）改 `bad` 三次 | 正確率與基準線各三組 |
| 二 | `my_confusion()` 兩行、`metrics()` 三行 | 兩個函式，後面一直用 |
| 三 | `plt.plot` 那一行 | `pr_curve.png` |
| 四 | 乾淨訓練集那一行 | 洩漏前後兩個分數 |
| 五（進階） | 找 F1 最高的門檻 | 最佳門檻與 F1 |

**每次只換一個變因**，才看得出是誰影響了分數。

### 第 1 格：造資料、切分、訓練

**這一格要做什麼**：沒有空格，只改 `bad` 那一行，`1`、`10`、`20` 三個值各跑一次，
每次都記下**正確率**與**全猜好品基準**兩個數字。

**`stratify=y` 在做什麼**：讓訓練與測試的類別比例一致。不加的話，
不平衡資料很容易切出一邊完全沒有壞品的測試集。

**寫對了會看到什麼**：兩個數字。**越接近，代表這個模型越沒有在做事。**

⚠️ `prob` 與 `pred` 這兩個變數後面每一格都會用到，**第 1 格沒跑成功後面全部會報錯**。

In [ ]:
#@title 第 1 格：造資料、切分、訓練
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
bad = 10                    # ← 改這裡：壞品佔百分之幾
X, y = make_classification(n_samples=4000, weights=[1 - bad/100],
                           n_informative=5, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=0)
m = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
prob = m.predict_proba(Xte)[:, 1]
pred = (prob >= 0.5).astype(int)
print("正確率 =", round(m.score(Xte, yte), 3))
print("全猜好品基準 =", round(1 - yte.mean(), 3))

### 第 2 格：自己寫混淆矩陣函式

**這一格要做什麼**：補兩行，**不准用 sklearn 的 `confusion_matrix`**。

規則只有一句：**第二個字母是模型說什麼，第一個字母是說得對不對。**

- `tp`：真的是 1，模型也說 1 → 兩個條件用 `&` 串起來。
- `fp`：真的是 0，模型卻說 1 → 把 `tp` 那行的第一個條件改成 `yt == 0` 就是了。

**為什麼要加 `int()`**：`sum()` 回傳的是 numpy 整數，轉成 `int` 印出來比較乾淨。

**為什麼括號要包三層**：`&` 的運算優先序比 `==` 高，不加括號會直接報錯
（先踩一次 `ValueError` 再回來看這句，印象最深）。

**寫對了會看到什麼**：四格加起來剛好 **1000**，和測試資料筆數一模一樣。
不等於就是條件寫反了。

⚠️ 這個函式後面三個任務都會一直用到，**不要關掉這一格**。

In [ ]:
#@title 第 2 格：自己寫 my_confusion()
def my_confusion(yt, yp):
    tp = ____   # ← 自己寫：真的是 1 且被判成 1 的筆數
    fp = ____   # ← 自己寫：真的是 0 卻被判成 1 的筆數
    fn = int(((yt == 1) & (yp == 0)).sum())
    tn = int(((yt == 0) & (yp == 0)).sum())
    return tp, fp, fn, tn

tp, fp, fn, tn = my_confusion(yte, pred)
print("TP", tp, " FP", fp)
print("FN", fn, " TN", tn)
print("四格加起來 =", tp + fp + fn + tn, "應等於", len(yte))

### 第 3 格：自己寫三個指標

**這一格要做什麼**：補三行，用上一格算出來的四格數字。

- `prec` 精確率：TP ÷ (TP + FP) —— 問「說是的準不準」。
- `rec` 召回率：TP ÷ (TP + FN) —— 問「該抓的抓到沒」。
- `f1`：2×prec×rec ÷ (prec + rec) —— **調和平均**，寫成 `(prec + rec) / 2` 是最常見的錯。

**驗算方法**：用課堂例題的 200 封信（TP=35、FP=15、FN=5、TN=145）代進去，
精確率 0.7、召回率 0.875，正確的 F1 是 **0.78**；如果你寫成算術平均會得到 0.79。

**寫對了會看到什麼**：四個介於 0 到 1 之間的數字。

⚠️ 門檻調到 0.9 時 `tp + fp` 可能等於 0，會出現 `ZeroDivisionError`——
這本身就是一個要寫進門檻紀錄表的觀察，不是你寫錯了。

In [ ]:
#@title 第 3 格：自己寫 metrics()
def metrics(tp, fp, fn, tn):
    acc = (tp + tn) / (tp + fp + fn + tn)
    prec = ____    # ← 自己寫：TP ÷ (TP + FP)
    rec = ____     # ← 自己寫：TP ÷ (TP + FN)
    f1 = ____      # ← 自己寫：2×prec×rec ÷ (prec + rec)
    return acc, prec, rec, f1

a, p, r, f = metrics(tp, fp, fn, tn)
print("正確率", round(a, 3), "精確率", round(p, 3))
print("召回率", round(r, 3), "F1", round(f, 3))

### 第 4 格：掃門檻並畫出 PR 曲線

**這一格要做什麼**：補 `plt.plot` 那一行——**x 放召回率、y 放精確率**。
放反了曲線方向會相反，正好可以當堂討論。

**為什麼軸標籤用英文**：Colab 預設字型沒有中文，中文標籤會變成一排方框。

**寫對了會看到什麼**：六列數字（門檻由低到高），加上一條**往右下掉**的曲線；
左邊檔案列表會多出一個 `pr_curve.png`。

**電腦教室才做得到的一步**：圖存好之後在左邊檔案列表按右鍵 → 下載，
存到自己的電腦再上傳繳交。

**門檻紀錄表（六列填滿才看得出趨勢，當堂收）**

| 判斷門檻 | 精確率 | 召回率 | 你看到什麼 |
|---|---|---|---|
| 0.1 | 　 | 　 | 門檻很低：幾乎全抓，誤報一大堆 |
| 0.3 | 　 | 　 | 抓得多，精確率開始往下掉 |
| 0.5 | 　 | 　 | 預設值，先記下來當基準 |
| 0.7 | 　 | 　 | 說是的比較準，但開始漏抓 |
| 0.9 | 　 | 　 | 可能幾乎都不抓，召回率接近 0 |
| 趨勢 | 往上 | 往下 | 門檻調高時，兩個指標的走向 |
| 你的結論 | 　 | 　 | 如果是疾病篩檢，你會選哪一個門檻 |

In [ ]:
#@title 第 4 格：掃門檻並畫出 PR 曲線
import matplotlib.pyplot as plt
ths, precs, recs = [], [], []
for th in [0.1, 0.2, 0.3, 0.5, 0.7, 0.9]:
    yp = (prob >= th).astype(int)
    tp, fp, fn, tn = my_confusion(yte, yp)
    a, p, r, f = metrics(tp, fp, fn, tn)
    ths.append(th); precs.append(p); recs.append(r)
    print(th, "精確率", round(p, 3), "召回率", round(r, 3))
plt.plot(____, ____, "o-")   # ← 自己寫：x 召回率、y 精確率
plt.xlabel("Recall"); plt.ylabel("Precision")
plt.savefig("pr_curve.png", dpi=120)
plt.show()

### 第 5 格：製造一次洩漏，再把它修好

**這一格要做什麼**：補 `k2` 那一行——改成**乾淨的**訓練集再比一次。

`Xbad` 把整份測試資料偷混進訓練集，`k1` 就是拿它訓練的。
1-NN 會直接記住每一筆資料，測試資料混進去以後，每一筆的最近鄰居就是它自己，
分數會**剛好 1.0000**。

**寫對了會看到什麼**：`有洩漏 = 1.0`、修好後大約 0.9 上下，
以及「分數虛高了」多少。**分數忽然變得很漂亮，先懷疑洩漏。**

**進階再問自己**：只混一半（`Xte[:500]`）會變幾分？
改成 `KNeighborsClassifier(15)` 又會怎樣？

In [ ]:
#@title 第 5 格：製造一次洩漏，再把它修好
from sklearn.neighbors import KNeighborsClassifier
import numpy as np
Xbad = np.vstack([Xtr, Xte])          # 偷混入整份測試資料
ybad = np.concatenate([ytr, yte])
k1 = KNeighborsClassifier(1).fit(Xbad, ybad)
k2 = KNeighborsClassifier(1).fit(____, ____)  # ← 自己寫
s1, s2 = k1.score(Xte, yte), k2.score(Xte, yte)
print("有洩漏 =", round(s1, 4), " 修好後 =", round(s2, 4))
print("分數虛高了", round(s1 - s2, 4))

### 第 6 格（進階，任務五）：自己找 F1 最高的門檻

**這一格要做什麼**：把第 4 格那六個門檻換成 **0.05 掃到 0.95**，
用 `for` 迴圈找出 **F1 最高**的門檻並印出來。要補三個空：

- 產生門檻清單：`np.arange(0.05, 1.0, 0.05)`。
- 比較與記錄：F1 比目前最好的還高就換掉。
- 印出最佳門檻與它的 F1。

**為什麼要 try**：門檻掃到很高時 `tp + fp` 會等於 0，
`metrics()` 會丟 `ZeroDivisionError`，用 `try` 直接跳過那個門檻就好。

**寫對了會看到什麼**：一個介於 0.1 到 0.6 之間的最佳門檻，
它的 F1 會比預設 0.5 那一個高一點點——**這就是「調門檻是最便宜的改善手段」**。

In [ ]:
#@title 第 6 格（進階）：自己找 F1 最高的門檻
# ←投影片未含，執行所需（投影片 LAB CHALLENGE B 的程式化版本）
best_th, best_f1 = None, -1
for th in ____:                 # ← 自己寫：np.arange(0.05, 1.0, 0.05)
    yp = (prob >= th).astype(int)
    try:
        a, p, r, f = metrics(*my_confusion(yte, yp))
    except ZeroDivisionError:
        continue                # 門檻太高，一個都沒抓，跳過
    if ____:                    # ← 自己寫：這次的 f 比 best_f1 還高
        best_th, best_f1 = th, f
print("F1 最高的門檻 =", round(____, 2))   # ← 自己寫：印出 best_th
print("那時候的 F1 =", round(best_f1, 3))

## 收工：延伸挑戰與繳交

- **A（每個人都要做完）換參數再跑**：把第 1 格的 `bad` 從 10 改成 1 再跑一次，
  記錄正確率與基準線的差距怎麼變。
- **B 自己找最佳門檻**：就是上面的第 6 格，做得完再往下做，做不完不影響及格。
- **C 說出為什麼**：說出訓練集混進測試資料為什麼會讓分數變高，並舉一個生活中的類比。

**電腦教室常見狀況**：
`NameError: prob` ＝第 1 格沒跑成功，回去重跑；
第 2 格報 `ValueError` ＝`&` 兩邊忘了加括號；
四格加起來不等於 1000 ＝判斷條件寫反了；
`ZeroDivisionError` ＝門檻太高一個都沒抓，這是觀察不是錯誤；
圖上中文變方框＝軸標籤改用英文。

In [ ]:
#@title 收工檢查（直接按 ▶）
# ←投影片未含，執行所需
print("本週要交：寫完的 .ipynb（六格全部跑過）、my_confusion 與 metrics 兩個函式")
print("門檻紀錄表（五個門檻各一列）、PR 曲線 pr_curve.png")
print("✅ 檔名：AI導論_W16_學號_姓名")
print("下週第 17 週：Keras 轉 TFLite 並量延遲。課前確認雲端硬碟還有空間")

---

<details>
<summary>參考解（七個空格都自己試過再打開）</summary>

```python
# 第 2 格
tp = int(((yt == 1) & (yp == 1)).sum())
fp = int(((yt == 0) & (yp == 1)).sum())

# 第 3 格
prec = tp / (tp + fp)
rec = tp / (tp + fn)
f1 = 2 * prec * rec / (prec + rec)

# 第 4 格
plt.plot(recs, precs, "o-")

# 第 5 格
k2 = KNeighborsClassifier(1).fit(Xtr, ytr)

# 第 6 格
for th in np.arange(0.05, 1.0, 0.05):
    ...
    if f > best_f1:
        best_th, best_f1 = th, f
print("F1 最高的門檻 =", round(best_th, 2))
```

為什麼是這樣寫：

- **`&` 的優先序比 `==` 高**，所以 `yt == 1 & yp == 1` 會先算 `1 & yp`，
  然後丟 `ValueError`。三層括號是必要的，不是為了好看。
- **F1 是調和平均**，不是算術平均。用一高一低的例子最清楚：
  精確率 1.0、召回率 0.01，算術平均有 0.5 看起來還行，
  調和平均只有 0.02——這才誠實。
- **PR 曲線 x 軸是召回率、y 軸是精確率**，因為習慣上「抓得多不多」放橫軸。
  放反了曲線會變成往右上，方向就錯了。
- **`k2` 要用 `Xtr, ytr`**：這才是沒看過測試資料的訓練集。
  `k1` 的分數剛好 1.0，是因為 1-NN 把每一筆測試資料都背起來了。
- **`np.arange` 回傳的是浮點數陣列**，所以 `best_th` 印出來要 `round`，
  否則會看到 0.30000000000000004 這種東西。

</details>